# 📘 Aula 06b: Redes Neurais com Keras & Scikit-Learn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula06b_keras.ipynb)

Seja muito bem-vindo a esta aula prática! Nas aulas anteriores, nós desmistificamos a matemática por trás do **Perceptron**, do **Adaline** e do **Multilayer Perceptron (MLP)** construindo suas operações vetoriais e o algoritmo de *Backpropagation* **do zero em NumPy** e conhecendo o **PyTorch**. 

Agora, daremos um passo fundamental na caixa de ferramentas do cientista de dados: aprenderemos a utilizar o **Keras** (a API de alto nível do TensorFlow para Deep Learning) integrada perfeitamente ao padrão consagrado do **Scikit-Learn** (`fit`, `predict`, `predict_proba`).

---

### 🎯 Objetivo da Aula
Construir classificadores neurais modernos com o ecossistema Keras/TensorFlow para o problema clássico de **reconhecimento de dígitos manuscritos (MNIST reduzido / Optical Recognition of Handwritten Digits)**, acompanhando a evolução arquitetural passo a passo:
1. **SLP (Single Layer Perceptron)**: Modelo linear sem camadas ocultas.
2. **SHLP (Single Hidden Layer Perceptron)**: Rede com uma camada oculta não-linear (`ReLU`).
3. **MLP (Multilayer Perceptron)**: Rede profunda com suporte a $N$ camadas ocultas arbitrárias.

---

### 💡 A Intenção Pedagógica do Professor
O Professor Boldt adota aqui um padrão de design profissional: encapsular modelos do Keras dentro de classes derivadas de `BaseEstimator` e `ClassifierMixin` do Scikit-Learn. Isso permite que qualquer modelo de Deep Learning seja utilizado diretamente com validação cruzada (`cross_val_score`), busca de hiperparâmetros (`GridSearchCV`) e pipelines sem precisar reescrever seu fluxo de dados!

---

### 📚 Glossário da Aula
* **Keras**: Biblioteca de alto nível para construção rápida e declarativa de redes neurais.
* **`keras.Sequential`**: Modelo linear em camadas (onde cada camada alimenta a subsequente sequencialmente).
* **`Dense`**: Camada totalmente conectada (*Fully Connected*), onde cada neurônio se conecta a todas as ativações da camada anterior.
* **`Softmax`**: Função de ativação para a camada de saída multiclasse que converte valores arbitrários (*logits*) em uma distribuição de probabilidades cuja soma é $1.0$.
* **`Categorical Crossentropy`**: Função de perda padrão para classificação multiclasse com alvos codificados em *one-hot*.
* **`to_categorical`**: Utilitário que converte rótulos inteiros de classes (ex: `0, 1, ..., 9`) em vetores binários esparsos (*one-hot vectors*).

---

### 🧭 Trilha da Aula
1. [Carregamento e Preparação do Dataset de Dígitos (load_digits)](#1-carregamento-e-divisão-dos-dados)
2. [SLP: Perceptron de Camada Única (Linear Multiclasse)](#2-slpclassifier-perceptron-de-camada-única)
3. [SHLP: Perceptron com Uma Camada Oculta Não-Linear](#3-shlpclassifier-uma-camada-oculta-com-relu)
4. [MLP: Rede Neural Multicamadas Genérica (N Camadas)](#4-mlpclassifier-arquitetura-profunda-modular)
5. [Experimentação Prática: Rede Profunda com Duas Camadas [50, 50]](#5-experimentação-profunda-com-duas-camadas-50-50)


## 1. Carregamento e Divisão dos Dados

O dataset `load_digits` do Scikit-Learn contém **1.797 imagens** de dígitos de $0$ a $9$ em escala de cinza de baixa resolução ($8 \times 8$ pixels = $64$ atributos por amostra).

> [!NOTE]
> Cada amostra é um vetor linearizado de dimensão $64$ (valores de intensidade de cinza de $0$ a $16$). O objetivo é mapear este vetor de $64$ entradas para uma das $10$ classes possíveis.

> [!IMPORTANT]
> Utilizamos `train_test_split` com `test_size=0.2` (80% treino, 20% teste) e fixamos a semente `random_state=42` para garantir reprodutibilidade exata dos experimentos.


In [ ]:
# Importa a função para carregar o conjunto de dados de dígitos manuscritos (8x8 pixels)
from sklearn.datasets import load_digits
# Importa a função utilitária para dividir os dados em partições de treino e teste
from sklearn.model_selection import train_test_split

# Carrega as características X (1797 amostras, 64 pixels cada) e os rótulos y (dígitos de 0 a 9)
X, y = load_digits(return_X_y=True)

# Divide o dataset: 80% para treino (X_train, y_train) e 20% para teste (X_test, y_test)
# random_state=42 garante que a divisão seja exatamente a mesma em qualquer execução
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 2. SLPClassifier: Perceptron de Camada Única

O **Single Layer Perceptron (SLP)** não possui camadas ocultas intermediárias. As $64$ entradas conectam-se diretamente aos $10$ neurônios de saída, aplicando uma função de ativação **Softmax**.

$$\hat{y} = \text{Softmax}(X \cdot W + b)$$

> [!NOTE]
> **O que faz o `to_categorical`?**
> Para treinar com a perda `categorical_crossentropy`, os rótulos devem ser codificados em *One-Hot*. Por exemplo, o dígito $3$ vira `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`.

> [!TIP]
> Herdando de `BaseEstimator` e `ClassifierMixin`, nosso estimador Keras ganha automaticamente os métodos do Scikit-Learn como `.score()`, clonagem e integração total a pipelines!


In [ ]:
# Classes base do Scikit-Learn para construir estimadores compatíveis
from sklearn.base import BaseEstimator, ClassifierMixin
# Métrica de acurácia para avaliação da taxa de acerto
from sklearn.metrics import accuracy_score
# Módulo Keras do TensorFlow para montagem da rede neural
from tensorflow import keras
# NumPy para manipulação matricial e operações vetoriais
import numpy as np

class SLPClassifier(BaseEstimator, ClassifierMixin):
  """
  Classificador Perceptron de Camada Única (Single Layer Perceptron).
  Conecta a camada de entrada diretamente aos neurônios de saída com ativação Softmax.
  """
  def __init__(self, max_iter=200):
    # Define o número máximo de épocas de treinamento
    self.max_iter = max_iter
    # Atributo que armazenará o modelo compilado do Keras
    self.model = None

  def fit(self, X, y):
    # Identifica os rótulos únicos de classes e mapeia para índices inteiros ordenados (0, 1, ..., C-1)
    self.labels, ids = np.unique(y, return_inverse=True)
    # Número de atributos de entrada (para dígitos: 64 features)
    input_shape = X.shape[1]
    # Número de classes de saída (para dígitos: 10 classes)
    output_shape = len(self.labels)
    # Converte os índices de classes para codificação One-Hot
    y_hot = keras.utils.to_categorical(ids, num_classes=output_shape)
    
    # Cria a arquitetura sequencial no Keras
    self.model = keras.Sequential([
        # Camada de entrada especificando o formato do vetor (64 atributos)
        keras.layers.Input(shape=(input_shape,)),
        # Camada densa com ativação softmax (gera probabilidades para cada classe)
        keras.layers.Dense(output_shape, activation='softmax')
    ])
    
    # Configura o algoritmo de otimização, função de perda e métrica de monitoramento
    self.model.compile(optimizer="adam",
                       loss="categorical_crossentropy",
                       metrics=["accuracy"])
    
    # Treina os pesos da rede por 'max_iter' épocas (verbose=0 omite o log por época)
    self.model.fit(X, y_hot, epochs=self.max_iter, verbose=0)
    return self

  def predict_proba(self, X):
    # Retorna a matriz de probabilidades [N, C] gerada pela função softmax
    return self.model.predict(X)

  def predict(self, X):
    # Obtém o índice da classe com maior probabilidade estimada para cada amostra
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    # Mapeia os índices de volta para os rótulos de classe originais
    return self.labels[y_pred]

# Instancia o modelo Perceptron de Camada Única
model = SLPClassifier()
# Ajusta os pesos com os dados de treino
model.fit(X_train, y_train)
# Realiza as predições no conjunto de teste independente
y_pred = model.predict(X_test)
# Calcula e exibe a acurácia obtida no conjunto de teste
print(f"Acurácia SLP (Camada Única): {accuracy_score(y_test, y_pred) * 100:.2f}%")


## 3. SHLPClassifier: Uma Camada Oculta com ReLU

O **Single Hidden Layer Perceptron (SHLP)** adiciona uma camada oculta intermediária de neurônios antes da saída.

$$\text{Entrada (64)} \longrightarrow \text{Oculta (100 neurônios, ReLU)} \longrightarrow \text{Saída (10 neurônios, Softmax)}$$

> [!NOTE]
> **Por que usar a ReLU (*Rectified Linear Unit*)?**
> A função $\text{ReLU}(z) = \max(0, z)$ introduz **não-linearidade** fundamental sem sofrer com a saturação ou desvanecimento do gradiente (*vanishing gradient*) que ocorria com a sigmoide e a tangente hiperbólica nas camadas intermediárias.

> [!TIP]
> Com 100 neurônios ocultos, a rede projeta o espaço de 64 pixels em um espaço latente de 100 dimensões, facilitando a separação linear das classes pela camada final Softmax.


In [ ]:
# Classes base do Scikit-Learn
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
from tensorflow import keras
import numpy as np

class SHLPClassifier(BaseEstimator, ClassifierMixin):
  """
  Classificador Neural com Uma Camada Oculta (Single Hidden Layer Perceptron).
  Utiliza ativação não-linear ReLU na camada intermediária e Softmax na saída.
  """
  def __init__(self, n_hidden=100, max_iter=200):
    # Número de neurônios na camada oculta
    self.n_hidden = n_hidden
    # Número de épocas de treinamento
    self.max_iter = max_iter
    self.model = None

  def fit(self, X, y):
    # Identifica classes únicas e índices inteiros
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1] # 64 atributos
    output_shape = len(self.labels) # 10 classes
    y_hot = keras.utils.to_categorical(ids, num_classes=output_shape)
    
    # Constrói a arquitetura Sequencial com 1 camada oculta
    self.model = keras.Sequential([
        keras.layers.Input(shape=(input_shape,)),
        # Camada Oculta: n_hidden neurônios com ativação ReLU (Rectified Linear Unit)
        keras.layers.Dense(self.n_hidden, activation='relu'),
        # Camada de Saída: output_shape neurônios com ativação Softmax para probabilidades
        keras.layers.Dense(output_shape, activation='softmax')
    ])
    
    # Compilação com otimizador Adam e perda Categorical Crossentropy
    self.model.compile(optimizer="adam",
                       loss="categorical_crossentropy",
                       metrics=["accuracy"])
    
    # Treinamento da rede
    self.model.fit(X, y_hot, epochs=self.max_iter, verbose=0)
    return self

  def predict_proba(self, X):
    return self.model.predict(X)

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

# Instancia o modelo com 1 camada oculta de 100 neurônios
model = SHLPClassifier()
# Treina o modelo nos dados de treino
model.fit(X_train, y_train)
# Faz predições no conjunto de teste
y_pred = model.predict(X_test)
# Avalia a nova taxa de acerto
print(f"Acurácia SHLP (1 Camada Oculta - 100 neurônios): {accuracy_score(y_test, y_pred) * 100:.2f}%")


## 4. MLPClassifier: Arquitetura Profunda Modular

O **Multilayer Perceptron (MLP)** genérico permite configurar **qualquer quantidade de camadas ocultas** com números arbitrários de neurônios, passando uma simples lista para o parâmetro `n_hidden` (ex: `n_hidden=[100]`, `n_hidden=[128, 64]`, etc.).

> [!NOTE]
> **Construção Dinâmica no Keras:**
> Em vez de passar uma lista estática para `keras.Sequential([...])`, instanciamos `model = keras.Sequential()` e usamos `model.add(...)` dinamicamente dentro de um laço de repetição `for n in self.n_hidden:`, permitindo profunda flexibilidade arquitetural.

> [!IMPORTANT]
> Essa classe replica de forma elegante e profissional em Keras a implementação matricial de $N$ camadas que construímos do zero em NumPy na Aula 05c!


In [ ]:
# Importação dos componentes Scikit-Learn e Keras
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
from tensorflow import keras
import numpy as np

class MLPClassifier(BaseEstimator, ClassifierMixin):
  """
  Classificador Multilayer Perceptron (MLP) modular e profundo.
  Permite configurar uma lista arbitrária de camadas ocultas através de 'n_hidden'.
  """
  def __init__(self, n_hidden=[100], max_iter=200):
    # Lista contendo o número de neurônios em cada camada oculta (ex: [100], [64, 32], etc.)
    self.n_hidden = n_hidden
    self.max_iter = max_iter
    self.model = None

  def fit(self, X, y):
    # Extrai classes únicas e índices inteiros
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1] # Dimensão de entrada (64 features)
    output_shape = len(self.labels) # Número de classes de saída (10 classes)
    y_hot = keras.utils.to_categorical(ids, num_classes=output_shape)
    
    # Inicializa o modelo sequencial vazio
    self.model = keras.Sequential()
    # Adiciona a camada de especificação de entrada
    self.model.add(keras.layers.Input(shape=(input_shape,)))
    
    # Itera sobre a lista de camadas ocultas, empilhando camadas Dense com ReLU dinamicamente
    for n in self.n_hidden:
      self.model.add(keras.layers.Dense(n, activation='relu'))
      
    # Adiciona a camada final de classificação Softmax
    self.model.add(keras.layers.Dense(output_shape, activation='softmax'))
    
    # Compila a rede com o otimizador Adam e perda de entropia cruzada categórica
    self.model.compile(optimizer="adam",
                       loss="categorical_crossentropy",
                       metrics=["accuracy"])
    
    # Executa o treinamento
    self.model.fit(X, y_hot, epochs=self.max_iter, verbose=0)
    return self

  def predict_proba(self, X):
    return self.model.predict(X)

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

# Instancia o modelo MLP com configuração padrão (1 camada oculta de 100 neurônios)
model = MLPClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Acurácia MLP Base ([100]): {accuracy_score(y_test, y_pred) * 100:.2f}%")


## 5. Experimentação Profunda com Duas Camadas: [50, 50]

Agora testamos o verdadeiro poder da arquitetura modular: instanciamos uma rede **mais profunda**, composta por duas camadas ocultas consecutivas com $50$ neurônios cada:

$$\text{Entrada (64)} \longrightarrow \text{Dense(50, ReLU)} \longrightarrow \text{Dense(50, ReLU)} \longrightarrow \text{Dense(10, Softmax)}$$

> [!TIP]
> Redes mais profundas conseguem compor representações hierárquicas mais ricas com menos parâmetros totais do que uma única camada gigantesca!


In [ ]:
# Instancia o MLP configurado com DUAS camadas ocultas profundas de 50 neurônios cada
model = MLPClassifier(n_hidden=[50, 50])
# Treina o modelo profundo
model.fit(X_train, y_train)
# Avalia no conjunto de teste
y_pred = model.predict(X_test)
# Exibe a acurácia final
print(f"Acurácia MLP Profundo ([50, 50]): {accuracy_score(y_test, y_pred) * 100:.2f}%")


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, realizamos a transição prática para a biblioteca **Keras/TensorFlow**, consolidando a ligação entre teoria de redes neurais e código de produção:

1. **Padrão Scikit-Learn no Keras**: Vimos como encapsular redes Keras em classes `BaseEstimator` e `ClassifierMixin`, permitindo que os métodos `fit`, `predict` e `predict_proba` funcionem em qualquer ecossistema de dados.
2. **Evolução Arquitetural**:
   * **SLP**: Modelo linear sem camadas ocultas $\rightarrow$ bom ponto de partida, mas limitado por fronteiras lineares.
   * **SHLP**: 1 Camada oculta com ativação $\text{ReLU}$ $\rightarrow$ projeta os dados em espaço de alta dimensão e eleva a acurácia.
   * **MLP Modular**: Empilhamento dinâmico de $N$ camadas através do loop `for n in self.n_hidden: model.add(...)`.
3. **Componentes Essenciais de Treinamento**:
   * Ativação final `Softmax` para probabilidades multiclasse.
   * Perda `categorical_crossentropy` combinada com rótulos `to_categorical`.
   * Otimizador adaptativo `Adam` para rápida convergência dos pesos.

---

## 👉 Próximo Passo
Na **Aula 06c**, aplicaremos redes neurais densas no processamento de linguagem natural e classificação multiclasse de textos jornalísticos com o dataset **Reuters**, lidando com vocabulários esparsos e dezenas de tópicos simultâneos!
